In [301]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [302]:
#prepare data
def prepare_data(df):
    # set all column name to lower case
    df.columns = df.columns.str.lower()
    # check if it contains date and close price column, if not, report error and exit the function
    if not 'date' in df.columns or not 'close' in df.columns:
        raise ValueError('DataFrame must have at least contain date and close price column')
    # rename the column name
    df = df.rename(columns={
        'date': 'Date',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume'
    })
    # if there is no Open, High, Low, fill it with close price
    if not 'Open' in df.columns:
        df['Open'] = df['Close']
    if not 'High' in df.columns:
        df['High'] = df['Close']
    if not 'Low' in df.columns:
        df['Low'] = df['Close']
    df['Date'] = pd.to_datetime(df['Date'])
    return df.set_index('Date')

In [303]:
# define a object called Strategy
class Strategy:
    def __init__(self):
        self.data = None
        self.position = 0
        self.orders = []
        self.indicators = {}
    def init(self):
        """Initialize strategy (pre-run)"""
        pass
    
    def next(self,i):
        """Main strategy logic"""
        pass
    
    def buy(self):
        self.orders.append(('buy', self.data.Close.iloc[-1]))
        self.position =1
        return self.position
    def sell(self):
        self.orders.append(('sell', self.data.Close.iloc[-1]))
        self.position =-1
        return self.position
    def I(self, func, *args, **kwargs):
        """添加指标计算"""
        indicator_name = f'{func.__name__}_{args}_{kwargs}'
        if indicator_name not in self.indicators:
            self.indicators[indicator_name] = func( *args)
        return self.indicators[indicator_name]

    # 添加技术指标函数
def SMA(series, window):
        return series.rolling(window).mean()
def crossover(a, b):
        """判断两个序列是否发生交叉"""
        return a.iloc[-2] < b.iloc[-2] and a.iloc[-1] > b.iloc[-1]


In [304]:
def backtest(df, strategy, commission=0.002, slippage=0.000, initial_cash=100000):
    strategy_instance = strategy()
    strategy_instance.data = df
    strategy_instance.init()
    #get the date index of df
    cash = initial_cash
    holdings = 0  # 持仓数量（正数表示多头，负数表示空头）
    equity = [initial_cash]
    positions = []  # 记录每日持仓方向
    
    for i in range(len(df)):
        current_price = df.iloc[i]['Close']
        
        # 调用策略获取信号
        signal = strategy_instance.next(i)
        
        # 保存当前持仓方向
        current_position = 1 if holdings > 0 else (-1 if holdings < 0 else 0)
        positions.append(current_position)
        
        # 执行交易逻辑
        if signal is not None and signal != current_position:
            # 先平仓
            if holdings > 0 and signal<=0:  # 平多仓
                close_price = current_price * (1 - slippage)
                cash += holdings * close_price * (1 - commission)
                holdings = 0
            elif holdings < 0 and signal>=0:  # 平空仓
                close_price = current_price * (1 + slippage)
                cash += holdings * close_price * (1 - commission)  # 注意holdings为负
                holdings = 0
                
            # 再开新仓
            if signal == 1:  # 开多仓
                buy_price = current_price * (1 + slippage)
                cost_per_share = buy_price * (1 + commission)
                shares = cash / cost_per_share
                holdings = shares
                cash = 0
            elif signal == -1:  # 开空仓
                sell_price = current_price * (1 - slippage)
                proceeds_per_share = sell_price * (1 - commission)
                shares = cash / (sell_price * (1 + commission))
                holdings = -shares
                cash += shares * proceeds_per_share
        
        # 计算当日权益（现金 + 持仓市值）
        position_value = holdings * current_price
        current_equity = cash + position_value
        equity.append(current_equity)
    
    # 计算统计指标
    returns = equity[-1] / initial_cash - 1
    equity_series = pd.Series(equity[1:], index=df.index)  # 移除初始值
    stats = {
        'final_equity': equity_series.iloc[-1] / initial_cash,
        'Return': returns,

    }
    return stats


In [305]:
# 纯双均值策略（0）
class SmaCross(Strategy):
    def init(self):
        price = self.data.Close
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)

    def next(self,i):
        if i < 20:
            return
        if crossover(self.ma1[:i-1], self.ma2[:i-1]):
            return self.buy()
        elif crossover(self.ma2[:i-1], self.ma1[:i-1]):
            return self.sell()

In [306]:
bu = pd.read_csv('data/bu.csv')
jd = pd.read_csv('data/jd.csv')
l = pd.read_csv('data/l.csv')
pp = pd.read_csv('data/pp.csv')
ru = pd.read_csv('data/ru.csv')
v = pd.read_csv('data/v.csv')
bu = prepare_data(bu)
jd = prepare_data(jd)
l = prepare_data(l)
pp = prepare_data(pp)
ru = prepare_data(ru)
v = prepare_data(v)
backtest(bu,SmaCross)

{'final_equity': np.float64(0.2988010914476727),
 'Return': np.float64(-0.7011989085523274)}

In [307]:
backtest(jd,SmaCross,)

{'final_equity': np.float64(0.24124914760286412),
 'Return': np.float64(-0.7587508523971359)}

In [308]:
backtest(l,SmaCross)

{'final_equity': np.float64(0.629051671443829),
 'Return': np.float64(-0.370948328556171)}

In [309]:
backtest(pp,SmaCross)

{'final_equity': np.float64(0.4779392870779057),
 'Return': np.float64(-0.5220607129220943)}

In [310]:
backtest(ru,SmaCross)

{'final_equity': np.float64(0.5113821356212903),
 'Return': np.float64(-0.4886178643787097)}

In [311]:
backtest(v,SmaCross)

{'final_equity': np.float64(0.8027529631417084),
 'Return': np.float64(-0.1972470368582916)}